# Projeto Final — Previsão de Risco de Fogo no Brasil

## Etapa 3 — Otimização, Interpretabilidade e Análise de Vieses

### 1. Introdução

Este notebook apresenta a terceira etapa do Projeto Final do MBA em Ciência de Dados e Inteligência Artificial.

O trabalho dá continuidade à preparação dos dados e à comparação dos modelos realizadas nas etapas anteriores.

Na Etapa 2, foram avaliados três algoritmos de classificação: Regressão Logística, Árvore de Decisão e Random Forest.

O Random Forest apresentou o melhor desempenho no conjunto de validação, com F1-macro de **0,5225**, sendo selecionado para otimização.

No conjunto de teste, o modelo inicial apresentou F1-macro de **0,4783**, evidenciando limitações de generalização.

### 2. Objetivos

Esta etapa tem três objetivos principais:

1. Otimizar os hiperparâmetros do Random Forest utilizando Optuna.
2. Investigar a contribuição dos atributos para as previsões utilizando SHAP.
3. Avaliar possíveis diferenças de desempenho entre grupos e regiões geográficas.

### 3. Metodologia

A otimização utilizará exclusivamente os conjuntos de treinamento e validação definidos na Etapa 2.

O F1-score macro permanecerá como métrica principal.

O conjunto de teste será preservado para uma avaliação final, sem utilizá-lo para escolher hiperparâmetros.

A interpretabilidade será utilizada para identificar os atributos mais relevantes e compreender suas contribuições para as classificações.

A análise de vieses investigará diferenças de desempenho entre grupos disponíveis na base.

### 4. Limitações

O projeto utiliza municípios-dia associados a focos detectados e valores válidos de um indicador de risco previamente calculado.

Portanto, os resultados não representam uma previsão validada da ocorrência de incêndios em todos os municípios brasileiros.

Também existem limitações relacionadas à cobertura temporal, à associação espacial das estações meteorológicas e à distribuição das categorias.

### 5. Resultado esperado

Ao final desta etapa, espera-se obter um modelo otimizado, apresentar os três atributos de maior contribuição para suas previsões e documentar possíveis diferenças de desempenho entre grupos.

Os resultados serão incorporados à documentação final do projeto.

In [1]:
# Bibliotecas necessárias
from pathlib import Path
import importlib.util
import pandas as pd
import numpy as np

# Diretório principal do projeto
projeto = Path.home() / "Sprint18_Projeto_Final"

# Localiza e carrega a base consolidada
arquivo_base = (
    projeto
    / "dados"
    / "processados"
    / "base_consolidada_v2.csv"
)

df_modelagem = pd.read_csv(
    arquivo_base,
    dtype={"id_municipio": "string"}
)

# Recupera os atributos territoriais
colunas_mapbiomas = [
    coluna
    for coluna in df_modelagem.columns
    if coluna.startswith("area_classe_")
]

# Reproduz a seleção de atributos da Etapa 2
colunas_modelo = [
    "temperatura_media",
    "umidade_minima",
    "precipitacao_24h",
    "horas_com_temperatura",
    "horas_com_umidade",
    "horas_com_precipitacao",
    "distancia_km",
    "possui_mapbiomas"
] + colunas_mapbiomas

# Separa atributos, alvo e datas
X = df_modelagem[colunas_modelo].copy()
y = df_modelagem["classe_risco"].copy()

datas = pd.to_datetime(df_modelagem["data"])

# Reproduz a divisão cronológica da Etapa 2
datas_unicas = sorted(datas.unique())

fim_treino = int(len(datas_unicas) * 0.60)
fim_validacao = int(len(datas_unicas) * 0.80)

inicio_validacao = datas_unicas[fim_treino]
inicio_teste = datas_unicas[fim_validacao]

mascara_treino = datas < inicio_validacao

mascara_validacao = (
    (datas >= inicio_validacao)
    & (datas < inicio_teste)
)

mascara_teste = datas >= inicio_teste

X_treino = X.loc[mascara_treino].copy()
X_validacao = X.loc[mascara_validacao].copy()
X_teste = X.loc[mascara_teste].copy()

y_treino = y.loc[mascara_treino].copy()
y_validacao = y.loc[mascara_validacao].copy()
y_teste = y.loc[mascara_teste].copy()

# Verifica a disponibilidade do Optuna
optuna_instalado = (
    importlib.util.find_spec("optuna") is not None
)

# Apresenta os resultados
print("ETAPA 3 — PREPARAÇÃO")

print("Dimensões da base:", df_modelagem.shape)
print("Atributos explicativos:", X.shape[1])

print("\nConjuntos:")
print("Treinamento:", len(X_treino))
print("Validação:", len(X_validacao))
print("Teste:", len(X_teste))

print("\nDatas de corte:")
print("Validação:", pd.Timestamp(inicio_validacao).date())
print("Teste:", pd.Timestamp(inicio_teste).date())

print("\nOptuna instalada:", optuna_instalado)
# As dependências devem ser instaladas previamente pelo requirements.txt.
# O Optuna será utilizado na etapa de otimização dos hiperparâmetros.

ETAPA 3 — PREPARAÇÃO
Dimensões da base: (19206, 45)
Atributos explicativos: 36

Conjuntos:
Treinamento: 12026
Validação: 3663
Teste: 3517

Datas de corte:
Validação: 2025-01-26
Teste: 2025-02-04

Optuna instalada: True


In [2]:
# Bibliotecas necessárias
import optuna

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

# Reduz as mensagens informativas
optuna.logging.set_verbosity(
    optuna.logging.WARNING
)

# Define a função que o Optuna tentará otimizar
def objetivo(trial):

    # Seleciona os hiperparâmetros
    parametros = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 50, 150, step=25
        ),
        "max_depth": trial.suggest_int(
            "max_depth", 5, 18
        ),
        "min_samples_leaf": trial.suggest_int(
            "min_samples_leaf", 5, 25
        ),
        "min_samples_split": trial.suggest_int(
            "min_samples_split", 2, 20
        ),
        "class_weight": "balanced",
        "random_state": 42,
        "n_jobs": -1
    }

    # Cria o pipeline
    modelo = Pipeline([
        ("imputacao", SimpleImputer(strategy="median")),
        ("modelo", RandomForestClassifier(**parametros))
    ])

    # Treina com os dados de treinamento
    modelo.fit(X_treino, y_treino)

    # Avalia somente na validação
    previsoes = modelo.predict(X_validacao)

    return f1_score(
        y_validacao,
        previsoes,
        average="macro",
        zero_division=0
    )


# Cria o estudo
estudo = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

# Executa os experimentos
estudo.optimize(
    objetivo,
    n_trials=20,
    show_progress_bar=True
)

# Apresenta os resultados
print("\nOTIMIZAÇÃO CONCLUÍDA")

print("Experimentos realizados:", len(estudo.trials))

print(
    f"Melhor F1-macro na validação: "
    f"{estudo.best_value:.4f}"
)

print("\nMelhores hiperparâmetros:")

for nome, valor in estudo.best_params.items():
    print(f"{nome}: {valor}")

print("\nReferência da Etapa 2: F1-macro = 0,5225")

  0%|          | 0/20 [00:00<?, ?it/s]


OTIMIZAÇÃO CONCLUÍDA
Experimentos realizados: 20
Melhor F1-macro na validação: 0.5383

Melhores hiperparâmetros:
n_estimators: 75
max_depth: 16
min_samples_leaf: 5
min_samples_split: 12

Referência da Etapa 2: F1-macro = 0,5225


In [3]:
# Cria o Random Forest otimizado
pipeline_otimizado = Pipeline([
    ("imputacao", SimpleImputer(strategy="median")),
    ("modelo", RandomForestClassifier(
        **estudo.best_params,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# Treina somente com os dados de treinamento
pipeline_otimizado.fit(X_treino, y_treino)

# Realiza previsões na validação
previsoes_otimizadas = pipeline_otimizado.predict(
    X_validacao
)

# Calcula as métricas
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    classification_report
)

acuracia_otimizada = accuracy_score(
    y_validacao,
    previsoes_otimizadas
)

f1_macro_otimizado = f1_score(
    y_validacao,
    previsoes_otimizadas,
    average="macro",
    zero_division=0
)

recall_alto_otimizado = recall_score(
    y_validacao,
    previsoes_otimizadas,
    labels=["Alto"],
    average="macro",
    zero_division=0
)

# Apresenta os resultados
print("RANDOM FOREST OTIMIZADO — VALIDAÇÃO")

print(f"Acurácia: {acuracia_otimizada:.4f}")
print(f"F1-macro: {f1_macro_otimizado:.4f}")
print(f"Recall Alto: {recall_alto_otimizado:.4f}")

print("\nRelatório de classificação:")

print(
    classification_report(
        y_validacao,
        previsoes_otimizadas,
        labels=["Baixo", "Médio", "Alto"],
        zero_division=0
    )
)

print("\nComparação do F1-macro:")
print("Random Forest original: 0.5225")
print(f"Random Forest otimizado: {f1_macro_otimizado:.4f}")

RANDOM FOREST OTIMIZADO — VALIDAÇÃO
Acurácia: 0.6931
F1-macro: 0.5383
Recall Alto: 0.6750

Relatório de classificação:
              precision    recall  f1-score   support

       Baixo       0.86      0.78      0.81      2346
       Médio       0.18      0.19      0.18       357
        Alto       0.57      0.68      0.62       960

    accuracy                           0.69      3663
   macro avg       0.53      0.55      0.54      3663
weighted avg       0.71      0.69      0.70      3663


Comparação do F1-macro:
Random Forest original: 0.5225
Random Forest otimizado: 0.5383


In [4]:
# Avaliação final do Random Forest otimizado no conjunto de teste

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    roc_auc_score,
    classification_report
)

# Realiza as previsões no conjunto de teste
previsoes_teste_otimizado = pipeline_otimizado.predict(X_teste)

# Calcula as métricas
acuracia_teste_otimizado = accuracy_score(
    y_teste, previsoes_teste_otimizado
)

f1_teste_otimizado = f1_score(
    y_teste,
    previsoes_teste_otimizado,
    average="macro"
)

recall_alto_teste_otimizado = recall_score(
    y_teste,
    previsoes_teste_otimizado,
    labels=["Alto"],
    average="macro",
    zero_division=0
)

# Apresenta os resultados
print("RANDOM FOREST OTIMIZADO — TESTE")

print(f"Acurácia: {acuracia_teste_otimizado:.4f}")
print(f"F1-macro: {f1_teste_otimizado:.4f}")
print(f"Recall Alto: {recall_alto_teste_otimizado:.4f}")

print("\nRelatório de classificação:")

print(
    classification_report(
        y_teste,
        previsoes_teste_otimizado,
        labels=["Baixo", "Médio", "Alto"],
        zero_division=0
    )
)

RANDOM FOREST OTIMIZADO — TESTE
Acurácia: 0.6514
F1-macro: 0.4771
Recall Alto: 0.5553

Relatório de classificação:
              precision    recall  f1-score   support

       Baixo       0.82      0.78      0.80      2309
       Médio       0.14      0.12      0.13       439
        Alto       0.46      0.56      0.50       769

    accuracy                           0.65      3517
   macro avg       0.47      0.49      0.48      3517
weighted avg       0.65      0.65      0.65      3517



### Avaliação final — Random Forest original versus otimizado

Após a otimização dos hiperparâmetros utilizando exclusivamente
os conjuntos de treinamento e validação, o Random Forest
otimizado foi avaliado no conjunto de teste.

A comparação apresentou os seguintes resultados:

| Métrica | RF original | RF otimizado |
|---|---:|---:|
| Acurácia | 0,6363 | 0,6514 |
| F1-macro | 0,4783 | 0,4771 |
| Recall Alto | 0,5618 | 0,5553 |

O modelo otimizado apresentou maior acurácia no teste,
mas não superou o modelo original em F1-macro ou recall
da categoria Alto.

Portanto, a melhoria de F1-macro observada na validação
(0,5225 para 0,5383) não se confirmou no teste.

Esse resultado evidencia limitações de generalização
e reforça a importância de avaliar modelos em dados
não utilizados na seleção de hiperparâmetros.

Como o F1-macro foi definido como métrica principal,
os resultados não justificam afirmar superioridade
do modelo otimizado sobre o original.

Nenhum ajuste adicional de hiperparâmetros foi realizado
com base nos resultados do conjunto de teste.

In [ ]:
# Importa as bibliotecas necessárias
import shap
import matplotlib.pyplot as plt

# Seleciona uma amostra reproduzível da validação
X_amostra = X_validacao.sample(
    n=100,
    random_state=42
)

# Recupera o imputador e o modelo treinado
imputador = pipeline_otimizado.named_steps["imputacao"]
floresta = pipeline_otimizado.named_steps["modelo"]

# Aplica o tratamento já ajustado no treinamento
X_amostra_tratada = pd.DataFrame(
    imputador.transform(X_amostra),
    columns=X_amostra.columns,
    index=X_amostra.index
)

# Cria o explicador SHAP
explicador = shap.TreeExplainer(floresta)

# Calcula os valores SHAP
valores_shap = explicador.shap_values(
    X_amostra_tratada
)

# Converte para array NumPy
valores_shap = np.asarray(valores_shap)

# Calcula a importância média absoluta
# Formato esperado: observações x atributos x classes
importancia_media = np.abs(valores_shap).mean(
    axis=(0, 2)
)

# Organiza as importâncias
df_importancia_shap = pd.DataFrame({
    "Atributo": X_amostra_tratada.columns,
    "Importância SHAP": importancia_media
}).sort_values(
    "Importância SHAP",
    ascending=False
)

# Apresenta os três atributos mais importantes
print("TRÊS ATRIBUTOS MAIS IMPORTANTES — SHAP")

display(
    df_importancia_shap.head(3).round(4)
)

# Cria o gráfico
top_10 = df_importancia_shap.head(10).sort_values(
    "Importância SHAP"
)

plt.figure(figsize=(9, 5))

plt.barh(
    top_10["Atributo"],
    top_10["Importância SHAP"]
)

plt.title("Random Forest otimizado — Importância global SHAP")
plt.xlabel("Média do valor absoluto de SHAP")
plt.tight_layout()
plt.show()

print("\nFormato dos valores SHAP:", valores_shap.shape)

### Interpretabilidade do modelo — Análise SHAP

#### Objetivo

Investigar quais atributos apresentam maior contribuição para as previsões do Random Forest otimizado.

#### Metodologia

Foi utilizado o `TreeExplainer` da biblioteca SHAP sobre uma amostra reproduzível de 100 observações do conjunto de validação.

A importância global foi calculada pela média dos valores absolutos de SHAP, considerando as três categorias de risco.

#### Resultados

Os três atributos de maior importância foram:

1. **`area_classe_39`:** importância SHAP de 0,0447.
2. **`temperatura_media`:** importância SHAP de 0,0418.
3. **`area_classe_3`:** importância SHAP de 0,0333.

Os resultados indicam que características territoriais e condições meteorológicas contribuíram para as classificações produzidas pelo modelo.

A temperatura média apresentou a segunda maior importância global, enquanto dois atributos territoriais do MapBiomas ocuparam a primeira e a terceira posições.

#### Limitações

Os códigos das classes territoriais foram preservados conforme a base original. Seu significado específico deverá ser confirmado na documentação correspondente antes de interpretações ambientais mais detalhadas.

A análise foi realizada com apenas 100 observações e seus resultados podem variar conforme a amostra selecionada.

Além disso, atributos correlacionados podem compartilhar suas contribuições nas explicações SHAP.

A importância global considera a magnitude das contribuições, mas não demonstra sua direção nem estabelece relações causais.

#### Conclusão

A análise SHAP identificou `area_classe_39`, `temperatura_media` e `area_classe_3` como os três atributos de maior contribuição média absoluta para as saídas do Random Forest otimizado na amostra avaliada.

Os resultados contribuem para a compreensão do comportamento do classificador, respeitando as limitações dos dados e do método.

### Interpretação de negócio — Resultados SHAP

A análise de importância global identificou três atributos de
maior contribuição para as previsões do Random Forest otimizado:

- `area_classe_39` — Soja: 0,0447.
- `temperatura_media` — Temperatura média: 0,0418.
- `area_classe_3` — Formação Florestal: 0,0333.

As classes 39 e 3 correspondem, respectivamente, a Soja e
Formação Florestal na legenda de referência do MapBiomas.

Os resultados sugerem que características de uso e cobertura
da terra, juntamente com condições meteorológicas, são
relevantes para as classificações produzidas pelo modelo.

Do ponto de vista operacional, essas informações podem
contribuir para compreender quais características ambientais
estão associadas às previsões e orientar investigações futuras.

Entretanto, a importância SHAP global representa a magnitude
das contribuições dos atributos. Ela não permite concluir,
isoladamente, se uma variável aumenta ou reduz o risco previsto.

Também não estabelece relações causais.

A coleção exata do MapBiomas utilizada na base original
não foi confirmada; portanto, as denominações das classes
foram interpretadas com base na legenda oficial de referência.

In [ ]:
# Define as faixas de distância
faixas = [0, 25, 50, float("inf")]

nomes_grupos = [
    "Até 25 km",
    "25 a 50 km",
    "Acima de 50 km"
]

# Organiza os dados de avaliação
df_vies_distancia = pd.DataFrame({
    "distancia_km": X_validacao["distancia_km"],
    "classe_real": y_validacao,
    "classe_prevista": previsoes_otimizadas
})

# Classifica as observações por distância
df_vies_distancia["grupo_distancia"] = pd.cut(
    df_vies_distancia["distancia_km"],
    bins=faixas,
    labels=nomes_grupos,
    include_lowest=True
)

# Armazena os resultados
resultados_grupos = []

for grupo in nomes_grupos:

    dados_grupo = df_vies_distancia[
        df_vies_distancia["grupo_distancia"] == grupo
    ]

    if len(dados_grupo) == 0:
        continue

    # Calcula o F1-macro
    f1_grupo = f1_score(
        dados_grupo["classe_real"],
        dados_grupo["classe_prevista"],
        labels=["Baixo", "Médio", "Alto"],
        average="macro",
        zero_division=0
    )

    # Calcula o recall da categoria Alto
    recall_alto_grupo = recall_score(
        dados_grupo["classe_real"],
        dados_grupo["classe_prevista"],
        labels=["Alto"],
        average="macro",
        zero_division=0
    )

    resultados_grupos.append({
        "Grupo": grupo,
        "Observações": len(dados_grupo),
        "Casos reais de Alto": (
            dados_grupo["classe_real"] == "Alto"
        ).sum(),
        "F1-macro": f1_grupo,
        "Recall Alto": recall_alto_grupo
    })

# Organiza a tabela
df_vies_resultados = pd.DataFrame(resultados_grupos)

print("ANÁLISE DE DESEMPENHO POR DISTÂNCIA")

display(df_vies_resultados.round(4))

# Verifica a cobertura da análise
print(
    "\nObservações avaliadas:",
    df_vies_resultados["Observações"].sum()
)

print(
    "Observações sem distância:",
    df_vies_distancia["grupo_distancia"].isna().sum()
)

In [ ]:
# Organiza os dados de validação
df_vies_meteorologia = X_validacao[
    [
        "temperatura_media",
        "umidade_minima",
        "precipitacao_24h"
    ]
].copy()

# Identifica observações com as três variáveis disponíveis
df_vies_meteorologia["grupo"] = np.where(
    df_vies_meteorologia[
        [
            "temperatura_media",
            "umidade_minima",
            "precipitacao_24h"
        ]
    ].notna().all(axis=1),
    "Três variáveis disponíveis",
    "Pelo menos uma ausente"
)

# Acrescenta as classificações
df_vies_meteorologia["classe_real"] = y_validacao
df_vies_meteorologia["classe_prevista"] = previsoes_otimizadas

# Calcula as métricas por grupo
resultados_meteorologia = []

for grupo, dados_grupo in df_vies_meteorologia.groupby("grupo"):

    resultados_meteorologia.append({
        "Grupo": grupo,
        "Observações": len(dados_grupo),
        "Casos reais de Alto": (
            dados_grupo["classe_real"] == "Alto"
        ).sum(),
        "F1-macro": f1_score(
            dados_grupo["classe_real"],
            dados_grupo["classe_prevista"],
            labels=["Baixo", "Médio", "Alto"],
            average="macro",
            zero_division=0
        ),
        "Recall Alto": recall_score(
            dados_grupo["classe_real"],
            dados_grupo["classe_prevista"],
            labels=["Alto"],
            average="macro",
            zero_division=0
        )
    })

# Apresenta os resultados
df_vies_meteorologia_resultados = pd.DataFrame(
    resultados_meteorologia
)

print("ANÁLISE POR DISPONIBILIDADE METEOROLÓGICA")

display(
    df_vies_meteorologia_resultados.round(4)
)

print(
    "\nTotal de observações:",
    df_vies_meteorologia_resultados["Observações"].sum()
)

## Análise de vieses e diferenças de desempenho entre grupos

### 1. Objetivo

Investigar possíveis diferenças de desempenho do Random Forest otimizado entre grupos de observações com distintas características geográficas e condições de disponibilidade meteorológica.

As análises foram realizadas com as 3.663 observações do conjunto de validação.

Foram utilizadas duas métricas: F1-score macro e recall da categoria Alto.

### 2. Desempenho por distância até a estação meteorológica

Os municípios foram agrupados em três faixas de distância até a estação meteorológica associada.

| Distância | Observações | Casos de risco Alto | F1-macro | Recall Alto |
|---|---:|---:|---:|---:|
| Até 25 km | 768 | 260 | 0,5189 | 0,6654 |
| 25 a 50 km | 1.539 | 481 | 0,5468 | 0,7110 |
| Acima de 50 km | 1.356 | 219 | 0,5216 | 0,6073 |

O grupo intermediário apresentou os maiores valores de F1-macro e recall Alto.

O grupo acima de 50 km apresentou o menor recall Alto, com diferença aproximada de 10,4 pontos percentuais em relação ao grupo intermediário.

Entretanto, não foi identificada uma redução progressiva do F1-macro conforme a distância aumentou.

Essas diferenças sugerem a necessidade de investigar a representatividade das medições meteorológicas em diferentes condições geográficas.

### 3. Desempenho por disponibilidade meteorológica

As observações foram divididas conforme a disponibilidade de temperatura média, umidade mínima e precipitação completa do dia anterior.

| Disponibilidade | Observações | Casos de risco Alto | F1-macro | Recall Alto |
|---|---:|---:|---:|---:|
| Pelo menos uma variável ausente | 2.084 | 545 | 0,5311 | 0,7229 |
| Três variáveis disponíveis | 1.579 | 415 | 0,5476 | 0,6120 |

O grupo com dados completos apresentou F1-macro ligeiramente superior, com diferença de 0,0165 ponto.

Por outro lado, o recall Alto foi aproximadamente 11,09 pontos percentuais maior no grupo com alguma variável meteorológica ausente.

Portanto, a disponibilidade completa das medições não esteve associada a desempenho superior em todas as métricas.

### 4. Limitações metodológicas

As análises foram realizadas em grupos observacionais, sem controle completo de possíveis diferenças regionais, temporais e de composição das categorias.

As observações também podem compartilhar estações meteorológicas e condições ambientais, não sendo necessariamente independentes.

As diferenças encontradas não demonstram causalidade nem permitem afirmar que a distância ou a ausência de medições sejam responsáveis diretamente pelos erros.

Além disso, os resultados foram calculados no conjunto de validação, utilizado anteriormente para selecionar os hiperparâmetros.

Por essa razão, devem ser interpretados como análises exploratórias do comportamento do modelo.

As comparações realizadas não constituem uma auditoria completa de equidade geográfica.

### 5. Conclusão

Foram identificadas diferenças de desempenho entre os grupos investigados, especialmente no recall da categoria Alto.

Os resultados indicam que a qualidade da classificação pode variar conforme características geográficas e de disponibilidade dos dados.

Essas diferenças merecem acompanhamento em futuras avaliações, preferencialmente com maior cobertura temporal e geográfica.

Não foram estabelecidas relações causais, nem demonstrada a existência ou ausência definitiva de vieses geográficos.

In [ ]:
# Analisa a distribuição regional da classe Alto na validação

from pathlib import Path
import pandas as pd

# Carrega somente as colunas necessárias
caminho_base = (
    Path.home()
    / "Sprint18_Projeto_Final"
    / "dados"
    / "processados"
    / "base_consolidada_v2.csv"
)

df_regional = pd.read_csv(
    caminho_base,
    usecols=["id_municipio", "data", "classe_risco"]
)

df_regional["data"] = pd.to_datetime(df_regional["data"])

# Identifica a UF pelos dois primeiros dígitos do código IBGE
df_regional["codigo_uf"] = df_regional["id_municipio"] // 100000

# Associa os códigos das UFs às regiões brasileiras
mapa_regioes = {}

for codigo in [11, 12, 13, 14, 15, 16, 17]:
    mapa_regioes[codigo] = "Norte"

for codigo in [21, 22, 23, 24, 25, 26, 27, 28, 29]:
    mapa_regioes[codigo] = "Nordeste"

for codigo in [31, 32, 33, 35]:
    mapa_regioes[codigo] = "Sudeste"

for codigo in [41, 42, 43]:
    mapa_regioes[codigo] = "Sul"

for codigo in [50, 51, 52, 53]:
    mapa_regioes[codigo] = "Centro-Oeste"

df_regional["regiao"] = df_regional["codigo_uf"].map(mapa_regioes)

# Seleciona o mesmo período de validação da Etapa 2
df_validacao_regional = df_regional[
    df_regional["data"].between("2025-01-26", "2025-02-03")
]

# Conta os casos reais da categoria Alto por região
casos_alto_regiao = (
    df_validacao_regional[
        df_validacao_regional["classe_risco"] == "Alto"
    ]
    .groupby("regiao")
    .size()
    .sort_values(ascending=False)
)

print("Total de observações na validação:", len(df_validacao_regional))
print("Total de casos reais Alto:", casos_alto_regiao.sum())

display(casos_alto_regiao.to_frame("Casos reais Alto"))

In [ ]:
# Avalia o desempenho do Random Forest otimizado por região

from sklearn.metrics import f1_score, recall_score

# Associa as regiões às observações da validação
df_desempenho_regional = pd.DataFrame({
    "regiao": df_regional.loc[y_validacao.index, "regiao"].to_numpy(),
    "classe_real": y_validacao.to_numpy(),
    "classe_prevista": pipeline_otimizado.predict(X_validacao)
})

resultados_regionais = []

for regiao, grupo in df_desempenho_regional.groupby("regiao"):

    f1 = f1_score(
        grupo["classe_real"],
        grupo["classe_prevista"],
        labels=["Baixo", "Médio", "Alto"],
        average="macro",
        zero_division=0
    )

    recall_alto = recall_score(
        grupo["classe_real"] == "Alto",
        grupo["classe_prevista"] == "Alto",
        zero_division=0
    )

    resultados_regionais.append({
        "Região": regiao,
        "Observações": len(grupo),
        "Casos Alto": (grupo["classe_real"] == "Alto").sum(),
        "F1-macro": round(f1, 4),
        "Recall Alto": round(recall_alto, 4)
    })

display(
    pd.DataFrame(resultados_regionais)
    .sort_values("Casos Alto", ascending=False)
)

### Análise complementar — Desempenho regional do modelo

Foi avaliado o desempenho do Random Forest otimizado nas cinco regiões brasileiras, utilizando o conjunto de validação.

A distribuição dos 960 casos reais da categoria Alto apresentou forte concentração no Nordeste, com 750 ocorrências, enquanto o Sudeste registrou apenas 13.

O recall da categoria Alto variou entre as regiões:

- **Nordeste:** 0,7240.
- **Centro-Oeste:** 0,5429.
- **Norte:** 0,5224.
- **Sul:** 0,5167.
- **Sudeste:** 0,0769.

O Nordeste apresentou o maior recall da categoria Alto, enquanto o Sudeste registrou o menor desempenho nessa métrica.

Entretanto, o número reduzido de observações reais da categoria Alto no Sudeste limita a estabilidade da estimativa e exige cautela nas comparações.

Os resultados evidenciam heterogeneidade regional no desempenho do classificador, mas não permitem afirmar que a localização geográfica seja a causa dessas diferenças.

A concentração das observações, as características dos dados e a cobertura meteorológica são fatores que devem ser considerados na interpretação.

**Conclusão:** o desempenho regional não é uniforme, sendo necessária cautela na utilização do modelo em regiões pouco representadas na validação.

## Conclusões — Etapa 3

### 1. Otimização de hiperparâmetros

Foi realizada a otimização do Random Forest utilizando a biblioteca Optuna, com 20 experimentos e F1-score macro como objetivo.

O modelo original apresentou F1-macro de 0,5225 no conjunto de validação.

Após a otimização, o F1-macro aumentou para 0,5383, representando uma melhoria absoluta de 0,0158 ponto.

Os hiperparâmetros selecionados foram:

- `n_estimators`: 75.
- `max_depth`: 16.
- `min_samples_leaf`: 5.
- `min_samples_split`: 12.

Apesar da melhoria no F1-macro e na acurácia, o recall da categoria Alto diminuiu de 0,6917 para 0,6750.

A otimização apresentou resultados favoráveis na validação, mas não demonstra, isoladamente, melhoria de generalização.

Na avaliação final no conjunto de teste, o Random Forest otimizado apresentou acurácia de **0,6514**, F1-macro de **0,4771** e recall da categoria Alto de **0,5553**. O modelo original apresentou, respectivamente, **0,6363**, **0,4783** e **0,5618**.

Portanto, embora a otimização tenha aumentado a acurácia, não houve melhoria no F1-macro ou no recall da categoria Alto no conjunto de teste. Considerando o F1-macro como métrica principal, os resultados não demonstram superioridade do modelo otimizado em relação ao original.

### 2. Interpretabilidade com SHAP

A análise SHAP foi realizada utilizando 100 observações do conjunto de validação.

Os três atributos de maior importância global foram:

1. `area_classe_39`: 0,0447.
2. `temperatura_media`: 0,0418.
3. `area_classe_3`: 0,0333.

Os resultados indicam a influência de características territoriais e meteorológicas nas previsões.

Entretanto, as importâncias SHAP representam contribuições para as saídas do modelo, não relações causais.

### 3. Análise de vieses e diferenças de desempenho

Foram investigadas possíveis diferenças de desempenho entre grupos definidos pela distância até a estação meteorológica e pela disponibilidade de variáveis meteorológicas.

Na análise geográfica, o recall Alto variou de 0,6073 a 0,7110.

Na análise de disponibilidade meteorológica, o recall Alto foi de 0,7229 entre observações com alguma variável ausente e de 0,6120 entre aquelas com as três variáveis disponíveis.

Essas diferenças indicam heterogeneidade de desempenho e justificam avaliações adicionais.

Não foi estabelecida relação causal entre as características dos grupos e os erros de classificação.

### 4. Limitações gerais

O projeto apresenta limitações importantes:

- A variável-alvo deriva de um indicador de risco previamente calculado.
- As observações estão condicionadas à existência de focos detectados.
- A cobertura temporal simultânea das fontes é limitada.
- As características territoriais utilizadas são históricas.
- A associação meteorológica depende da proximidade entre municípios e estações.
- O desempenho da categoria Médio permanece reduzido.
- A otimização e as análises de grupos utilizaram um período de validação relativamente curto.

Portanto, o modelo não deve ser interpretado como um sistema operacional validado de previsão de incêndios.

### 5. Considerações finais

A Etapa 3 demonstrou a aplicação de técnicas de otimização, interpretabilidade e investigação de diferenças de desempenho em um problema de classificação multiclasse.

O Random Forest otimizado apresentou melhoria no F1-macro de validação, e a análise SHAP permitiu identificar os atributos de maior contribuição para suas previsões.

As comparações entre grupos revelaram diferenças que merecem investigação em estudos futuros.

Os resultados constituem uma avaliação experimental, restrita aos dados e às condições metodológicas utilizadas.

A eventual aplicação operacional exigiria dados mais abrangentes, validação externa e avaliação independente do desempenho em diferentes períodos e regiões.

In [ ]:
from pathlib import Path
import subprocess

# Diretório principal do projeto
projeto = Path.home() / "Sprint18_Projeto_Final"

# Arquivos revisados
arquivos_git = [
    "notebooks/Sprint18_Etapa1_Entrega.ipynb",
    "notebooks/Sprint18_Etapa2_Modelagem.ipynb",
    "notebooks/Sprint18_Etapa3_Otimizacao_Interpretabilidade.ipynb"
]

# Confirma que todos os arquivos existem
for nome in arquivos_git:
    caminho = projeto / nome

    if not caminho.is_file():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")

# Adiciona exclusivamente os três notebooks
subprocess.run(
    ["git", "add", "--"] + arquivos_git,
    cwd=projeto,
    check=True
)

# Verifica quais alterações foram preparadas
resultado = subprocess.run(
    ["git", "diff", "--cached", "--name-only"],
    cwd=projeto,
    capture_output=True,
    text=True,
    check=True
)

print("Arquivos preparados para publicação:")
print(resultado.stdout)

print("Verificação concluída com sucesso!")

In [ ]:
# Registra as versões revisadas
resultado_commit = subprocess.run(
    [
        "git", "commit",
        "-m", "Sprint 18 - Revisao final dos notebooks"
    ],
    cwd=projeto,
    capture_output=True,
    text=True
)

print("COMMIT")
print("Código de retorno:", resultado_commit.returncode)
print(resultado_commit.stdout)

if resultado_commit.returncode != 0:
    print(resultado_commit.stderr)
    raise RuntimeError("Commit não concluído.")

# Publica as alterações no GitHub
resultado_push = subprocess.run(
    ["git", "push", "origin", "main"],
    cwd=projeto,
    capture_output=True,
    text=True
)

print("\nPUBLICAÇÃO NO GITHUB")
print("Código de retorno:", resultado_push.returncode)
print(resultado_push.stdout)
print(resultado_push.stderr)

if resultado_push.returncode != 0:
    raise RuntimeError("Publicação não concluída.")

In [ ]:
from pathlib import Path

# Localiza o README
arquivo_readme = projeto / "README.md"

# Lê o conteúdo
conteudo = arquivo_readme.read_text(encoding="utf-8")

# Atualiza as referências ao status do projeto
conteudo = conteudo.replace(
    "Etapa 1 — Preparação dos dados e análise exploratória.\n\nProjeto em desenvolvimento.",
    "Etapas 1, 2 e 3 — Desenvolvimento analítico concluído.\n\n"
    "Revisão de reprodutibilidade e submissão acadêmica pendentes."
)

# Acrescenta o status final de forma explícita
secao_status = """
## 20. Estado atual do projeto

As três etapas analíticas previstas foram desenvolvidas:

- Etapa 1: preparação, integração e análise exploratória.
- Etapa 2: modelagem, avaliação e planejamento experimental.
- Etapa 3: otimização, interpretabilidade e análise de vieses.

Os notebooks revisados estão disponíveis no repositório.

A reprodução integral do fluxo a partir dos dados originais
ainda não foi comprovada em um ambiente independente.

Os resultados devem ser interpretados considerando as
limitações metodológicas documentadas.
"""

if "## 20. Estado atual do projeto" not in conteudo:
    conteudo += "\n" + secao_status.strip() + "\n"

# Salva a atualização
arquivo_readme.write_text(conteudo, encoding="utf-8")

print("README atualizado com sucesso!")
print("Seção de status encontrada:",
      "## 20. Estado atual do projeto" in conteudo)

In [ ]:
import subprocess

# Prepara somente o README
subprocess.run(
    ["git", "add", "README.md"],
    cwd=projeto,
    check=True
)

# Registra a atualização
commit = subprocess.run(
    [
        "git", "commit", "-m",
        "Atualiza documentacao final do Sprint 18"
    ],
    cwd=projeto,
    capture_output=True,
    text=True
)

print("COMMIT")
print("Código de retorno:", commit.returncode)
print(commit.stdout)

if commit.returncode != 0:
    print(commit.stderr)
    raise RuntimeError("Commit não concluído.")

# Publica no GitHub
push = subprocess.run(
    ["git", "push", "origin", "main"],
    cwd=projeto,
    capture_output=True,
    text=True
)

print("\nPUBLICAÇÃO")
print("Código de retorno:", push.returncode)
print(push.stderr)

if push.returncode != 0:
    raise RuntimeError("Publicação não concluída.")

In [ ]:
from pathlib import Path

projeto = Path.home() / "Sprint18_Projeto_Final"

arquivo_base = (
    projeto
    / "dados"
    / "processados"
    / "base_consolidada_v2.csv"
)

print("Arquivo encontrado:", arquivo_base.is_file())

if arquivo_base.is_file():
    tamanho_mb = arquivo_base.stat().st_size / (1024 ** 2)

    print(f"Tamanho: {tamanho_mb:.2f} MB")
    print("Localização:", arquivo_base)

In [ ]:
import pandas as pd

# Consulta somente os nomes das colunas
colunas_base = pd.read_csv(
    arquivo_base,
    nrows=0
).columns.tolist()

print("Quantidade de colunas:", len(colunas_base))

print("\nColunas da base:")
for coluna in colunas_base:
    print("-", coluna)

In [ ]:
import subprocess

# Arquivo processado necessário às Etapas 2 e 3
arquivo_git = "dados/processados/base_consolidada_v2.csv"

# Adiciona somente o arquivo selecionado
subprocess.run(
    ["git", "add", "--", arquivo_git],
    cwd=projeto,
    check=True
)

# Confirma os arquivos preparados
resultado = subprocess.run(
    ["git", "diff", "--cached", "--name-only"],
    cwd=projeto,
    capture_output=True,
    text=True,
    check=True
)

print("Arquivos preparados para publicação:")
print(resultado.stdout)

In [ ]:
# Registra a base consolidada no histórico local do Git
resultado_commit = subprocess.run(
    [
        "git",
        "commit",
        "-m",
        "Adiciona base consolidada para reproducao dos modelos"
    ],
    cwd=projeto,
    capture_output=True,
    text=True
)

print("COMMIT")
print("Código de retorno:", resultado_commit.returncode)
print(resultado_commit.stdout)

if resultado_commit.stderr:
    print(resultado_commit.stderr)

In [ ]:
# Localiza o README
arquivo_readme = projeto / "README.md"

# Lê o conteúdo atual
conteudo = arquivo_readme.read_text(encoding="utf-8")

# Documenta a base consolidada
secao_dados = """
## 21. Base consolidada para reprodução

As Etapas 2 e 3 utilizam o arquivo:

`dados/processados/base_consolidada_v2.csv`

A base contém 19.206 observações município-dia e 45 colunas.

O arquivo foi produzido durante a Etapa 1 a partir da
integração de dados de queimadas, meteorologia e MapBiomas.

### Reprodução das Etapas 2 e 3

1. Obter o repositório do projeto.
2. Manter a estrutura original de diretórios.
3. Abrir os notebooks em um ambiente Jupyter com as
   bibliotecas necessárias instaladas.
4. Executar o notebook da Etapa 2.
5. Executar o notebook da Etapa 3.

Os notebooks utilizam o caminho relativo à pasta
`Sprint18_Projeto_Final/dados/processados/`.

A base consolidada permite executar as análises das
Etapas 2 e 3 sem repetir o processamento dos
arquivos originais.

### Limitações

A base foi construída com municípios-dia associados
a focos detectados.

Sua utilização não permite concluir que o modelo
prevê a ocorrência de incêndios em todos os
municípios brasileiros.

A reprodução integral da Etapa 1 exige os arquivos
originais, que não estão incluídos no repositório.
"""

# Evita duplicação
if "## 21. Base consolidada para reprodução" not in conteudo:

    conteudo += "\n" + secao_dados.strip() + "\n"

    arquivo_readme.write_text(
        conteudo,
        encoding="utf-8"
    )

    print("README atualizado com a base consolidada!")

else:
    print("Seção já existente. Nenhuma alteração realizada.")

print(
    "Documentação encontrada:",
    "## 21. Base consolidada para reprodução"
    in arquivo_readme.read_text(encoding="utf-8")
)